# Run Inference (no terminal needed)

This notebook runs the detection pipeline on a whole dataset and writes a
COCO-format result JSON, exactly like the `examples/*.py` launchers do - but
without touching a terminal.

## How to use

1. Pick the "Zero-shot Insect Detection" kernel (menu: Kernel -> Change kernel).
2. Edit the config block in cell [2]: point `COCO_JSON_PATH` / `IMAGES_DIR` at
   your dataset (or set `COCO_JSON_PATH = None` to just scan an image folder,
   no annotations needed), pick a detector in `MODEL`, and adjust thresholds.
3. Run all cells (menu: Run -> Run All Cells).

The result JSON is written to `OUTPUT_PATH`. Open `examples/eval.ipynb`
next to evaluate it against the COCO ground truth.

## Before you start

- Weight downloads (YOLO baselines, binary classifier) are documented in
  `MODEL_WEIGHTS.md`; `bash scripts/download_weights.sh` fetches everything.
- Zero-shot detectors need a GPU for reasonable speed (at least 8 GB VRAM);
  YOLO baselines run fine on CPU. See "Hardware expectations" in the README.
- SAM3 and DINOv3 need `HF_TOKEN` set in `.env`.


In [1]:
import os
from pathlib import Path

# Repo root discovery: works whether the kernel cwd is the repo root or examples/
ROOT = Path.cwd()
while not (ROOT / "utils" / "models.py").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if ROOT == ROOT.parent:  # walked up to the filesystem root
    raise RuntimeError(
        "Repo root not found. Open this notebook from inside the "
        "zero-shot-insect-detection repo directory."
    )

os.environ.setdefault("MODEL_CACHE_DIR", str(ROOT / "models-cache"))

from utils.models import (
    Bjerge2023Detector, FlatBugDetector, GroundingDINODetector,
    InsectsDCTDetector, Marin2025Detector, OwlDetector, SAM3Detector,
    Stark2023Detector,
)
from utils.inference import PROMPTS, run_inference, has_binary_classifier


/home/lollogiro/Documenti/UNI/master_thesis_project/zero-shot-insect-detection/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ---------------------------------------------------------------------------
# Config: same options as the examples/*.py launchers
# ---------------------------------------------------------------------------
MODEL = "marin2025"  # default: YOLO baseline, fast on CPU (42 MB download)
                     # other options: grounding_dino, sam3, owl (zero-shot,
                     # need a GPU), stark2023, bjerge2023, flatbug, insectsdct
COCO_JSON_PATH = str(ROOT / "example_dataset" / "detections_val_20.json")
IMAGES_DIR = str(ROOT / "example_dataset")
OUTPUT_PATH = str(ROOT / "results" / "my_run.json")
DESCRIPTION = "My run"

# Text prompt for zero-shot detectors (ignored by YOLO models):
# P1 = just "insect", P2 = contextual phrases, P3 = common insect names
TARGETS = PROMPTS["P3"]
NEGATIVE_PROMPT = None

CONF_THRESHOLD = 0.1
IOU_THRESHOLD = 0.1
SLICE_SIZE = None      # None = whole image; set e.g. 1024 for large images
OVERLAP = 0.2

# Binary classifier (optional): insect vs background filter
USE_BINARY_CLASSIFIER = False
DISCRIMINATOR_MODE = "mlp"        # "mlp" or "svm"
DISCRIMINATOR_NAME = "dinov3"     # "dinov3" or "bioclip2_5"
BINARY_SCORE_MODE = "joint"       # "joint" (product) or "mean" (average)
FIXRES = 0.3                      # 30% margin around the box before classifying
DISCRIMINATOR_FILTERING = None    # drop detections below this classifier score


In [3]:
# ---------------------------------------------------------------------------
# Launch
# ---------------------------------------------------------------------------
YOLO_MODELS = {"marin2025", "stark2023", "bjerge2023", "flatbug", "insectsdct"}

DETECTORS = {
    "grounding_dino": lambda: GroundingDINODetector(
        model_name="IDEA-Research/grounding-dino-base"),
    "sam3": lambda: SAM3Detector(),
    "owl": lambda: OwlDetector(),
    "marin2025": lambda: Marin2025Detector(
        model_name=str(ROOT / "models-cache/marin2025.pt")),
    "stark2023": lambda: Stark2023Detector(
        model_name=str(ROOT / "models-cache/yolov5_s_best_stark2023.pt")),
    "bjerge2023": lambda: Bjerge2023Detector(
        model_name=str(ROOT / "models-cache/insect1201-bestF1-1280v5m6_bjerge2023.pt")),
    "flatbug": lambda: FlatBugDetector(
        model_name=str(ROOT / "models-cache/flat_bug_M.pt")),
    "insectsdct": lambda: InsectsDCTDetector(
        model_name=str(ROOT / "models-cache/insectsDCT_insects6ColorExtend.pt")),
}

if MODEL not in DETECTORS:
    raise ValueError(f"Unknown MODEL {MODEL!r}; choose one of {sorted(DETECTORS)}")

detector = DETECTORS[MODEL]()
print("Running:", detector)

# Binary classifier: active only if enabled and weights are present
if USE_BINARY_CLASSIFIER and has_binary_classifier(DISCRIMINATOR_MODE, DISCRIMINATOR_NAME):
    discriminator_mode = DISCRIMINATOR_MODE
    discriminator_name = DISCRIMINATOR_NAME
    print(f"Binary classifier: {DISCRIMINATOR_MODE} / {DISCRIMINATOR_NAME}")
else:
    discriminator_mode = None
    discriminator_name = None

# Prompt is ignored by supervised (YOLO) detectors
prompt = None if MODEL in YOLO_MODELS else TARGETS

result = run_inference(
    detector=detector,
    coco_json_path=COCO_JSON_PATH,
    images_dir=IMAGES_DIR,
    prompt=prompt,
    negative_prompt=NEGATIVE_PROMPT,
    output_path=OUTPUT_PATH,
    description=DESCRIPTION,
    conf_threshold=CONF_THRESHOLD,
    iou_threshold=IOU_THRESHOLD,
    slice_size=SLICE_SIZE,
    overlap=OVERLAP,
    discriminator_mode=discriminator_mode,
    discriminator_name=discriminator_name,
    fixres=FIXRES,
    discriminator_filtering=DISCRIMINATOR_FILTERING,
    binary_score_mode=BINARY_SCORE_MODE,
)
print("Done. Result JSON:", OUTPUT_PATH)


YOLOv5 🚀 2026-8-10 Python-3.13.11 torch-2.13.0+cu130 CPU

Fusing layers... 
Model summary: 90 layers, 20,852,934 parameters, 0 gradients, 47.9 GFLOPs
Adding AutoShape... 
Running: Marin2025Detector(model='/home/lollogiro/Documenti/UNI/master_thesis_project/zero-shot-insect-detection/models-cache/marin2025.pt', supports_mask=False)
Found 9 images to process.


[/home/lollogiro/Documenti/UNI/master_thesis_project/zero-shot-insect-detection/models-cache/marin2025.pt] inference: 100%|██████████| 9/9 [00:02<00:00,  3.46it/s]


Done in 2.6s.  5 detections saved to: /home/lollogiro/Documenti/UNI/master_thesis_project/zero-shot-insect-detection/results/my_run.json
Done. Result JSON: /home/lollogiro/Documenti/UNI/master_thesis_project/zero-shot-insect-detection/results/my_run.json


## Next steps

- **Evaluate this run**: open `examples/eval.ipynb` with the same kernel,
  set `OUTPUT_PATH` in its config cell to `results/my_run.json` (or keep the
  default: it falls back to a shipped example run when no local result exists)
  and run all cells: COCO mAP/mAR, F1-vs-confidence and FROC curves,
  class-aware recall, visual inspection.
- **First time?** If a model fails with a missing-weights error, run
  `bash scripts/download_weights.sh` or follow `MODEL_WEIGHTS.md`.
- **Large images**: set `SLICE_SIZE = 1024` (and `OVERLAP` ~0.2) to process
  big camera-trap images in tiles; detections across tiles are fused with NMS.
